# 🛡️ NLP Capstone Project: SMS Spam Detection
**Project Name:** Sentinel-SMS  
**Objective:** Build an end-to-end Machine Learning & NLP pipeline to detect whether an incoming SMS is **Spam** or **Ham (Legitimate)**, and deploy it as a functional web application.

---
### Stages Covered:
1. Problem Identification & Dataset Loading
2. Data Cleaning & Handling Inconsistencies
3. Exploratory Data Analysis (EDA) & Visualizations
4. Text Preprocessing & NLP Normalization (Tokenization, Lemmatization, Stopwords)
5. Feature Engineering (TF-IDF N-Grams + Handcrafted Metadata Signals)
6. Model Development & 5-Fold Cross-Validation (Naive Bayes, Logistic Regression, Linear SVM, Random Forest)
7. Comprehensive Evaluation (Accuracy, Precision, Recall, F1-Score, Confusion Matrix, ROC-AUC)
8. Model Persistence & Live Test Inferences

## 1. Import Libraries & Set Up Environment

In [ ]:
import os
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import joblib

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
from scipy.sparse import hstack, csr_matrix

# Ensure local project paths for NLTK
nltk_path = os.path.join(os.getcwd(), 'nltk_data')
if os.path.exists(nltk_path) and nltk_path not in nltk.data.path:
    nltk.data.path.insert(0, nltk_path)

print('All libraries imported successfully!')

## 2. Dataset Collection & Inspection
We load the **UCI SMS Spam Collection** dataset containing 5,572 SMS messages labeled as either `ham` or `spam`.

In [ ]:
data_path = os.path.join('data', 'SMSSpamCollection')
df = pd.read_csv(data_path, sep='\t', header=None, names=['label', 'message'], encoding='latin-1')
df = df[['label', 'message']].dropna()
df['label'] = df['label'].str.strip().str.lower()
df['label_num'] = (df['label'] == 'spam').astype(int)

print(f'Total messages: {len(df):,}')
print(f'Spam count: {df["label_num"].sum():,} ({df["label_num"].mean()*100:.2f}%)')
print(f'Ham count: {(df["label_num"]==0).sum():,} ({(1-df["label_num"].mean())*100:.2f}%)')
df.head()

## 3. Data Cleaning & Feature Engineering
Spam messages frequently feature specific structural properties:
- Higher character length
- Higher digit and uppercase density
- Presence of URLs and currency symbols (£, $, €)
- Excessive punctuation (exclamation marks)

We extract these as numerical features before text normalization.

In [ ]:
def add_engineered_features(df):
    data = df.copy()
    data['msg_length'] = data['message'].str.len()
    data['word_count'] = data['message'].str.split().str.len()
    data['num_digits'] = data['message'].str.count(r'\d')
    data['num_uppercase'] = data['message'].apply(lambda x: sum(1 for c in x if c.isupper()))
    data['has_url'] = data['message'].str.contains(r'http|www', case=False, regex=True).astype(int)
    data['has_currency'] = data['message'].str.contains(r'[£$€]', regex=True).astype(int)
    data['num_excl'] = data['message'].str.count(r'!')
    return data

df = add_engineered_features(df)
df.groupby('label')[['msg_length', 'num_digits', 'num_uppercase', 'has_url', 'has_currency', 'num_excl']].mean()

## 4. Text Preprocessing & Lemmatization

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_message(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' url ', text)
    text = re.sub(r'\b\d{10,}\b|\b\d[\d\s\-]{7,}\d\b', ' phone ', text)
    text = re.sub(r'[£$€]\s?\d+(\.\d+)?', ' money ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['message'].apply(clean_message)
df[['message', 'clean_text', 'label']].head(6)

## 5. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
sns.countplot(x='label', data=df, palette=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Class Counts (Ham vs Spam)')

# Length distribution comparison
sns.histplot(df[df['label']=='ham']['msg_length'], bins=50, color='#2ecc71', label='Ham', kde=True, ax=axes[1])
sns.histplot(df[df['label']=='spam']['msg_length'], bins=50, color='#e74c3c', label='Spam', kde=True, ax=axes[1])
axes[1].set_title('Message Length Distribution')
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Feature Extraction: TF-IDF + Metadata Matrix
We transform cleaned text into sublinear TF-IDF word unigrams and bigrams, concatenated with the engineered features.

In [ ]:
X_train_df, X_test_df, y_train, y_test = train_test_split(
    df, df['label_num'], test_size=0.2, random_state=42, stratify=df['label_num']
)

vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

tfidf_train = vectorizer.fit_transform(X_train_df['clean_text'])
tfidf_test = vectorizer.transform(X_test_df['clean_text'])

num_cols = ['msg_length', 'word_count', 'num_digits', 'num_uppercase', 'has_url', 'has_currency', 'num_excl']
num_train = csr_matrix(X_train_df[num_cols].values)
num_test = csr_matrix(X_test_df[num_cols].values)

X_train = hstack([tfidf_train, num_train])
X_test = hstack([tfidf_test, num_test])

print(f'Feature matrix shape (Train): {X_train.shape}')
print(f'Feature matrix shape (Test): {X_test.shape}')

## 7. Model Training & 5-Fold Cross-Validation Benchmark
We benchmark 4 diverse algorithms using 5-fold Stratified Cross-Validation on the training set.

In [ ]:
classifiers = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Logistic Regression': LogisticRegression(C=5, max_iter=1000),
    'Linear SVM': LinearSVC(C=1.0, max_iter=2000),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    cv_results[name] = scores.mean()
    print(f'{name:<25} Mean F1: {scores.mean():.4f} (+/- {scores.std():.4f})')

best_name = max(cv_results, key=cv_results.get)
best_model = classifiers[best_name]
print(f'\nSelected Best Model: {best_name}')

## 8. Final Model Training & Test Set Evaluation
We train the selected champion model on the full training set and evaluate on the untouched test partition (1,115 messages).

In [ ]:
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

if hasattr(best_model, 'predict_proba'):
    y_prob = best_model.predict_proba(X_test)[:, 1]
else:
    y_prob = best_model.decision_function(X_test)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f'Final Test Accuracy: {acc*100:.2f}%')
print(f'Final ROC-AUC: {auc:.4f}\n')
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'], ax=axes[0])
axes[0].set_title(f'Confusion Matrix ({best_name})')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.4f})', color='darkorange', lw=2)
axes[1].plot([0, 1], [0, 1], color='navy', linestyle='--')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 9. Live Inference Demonstration
Testing custom incoming messages against the trained NLP pipeline.

In [ ]:
def test_sms(msg):
    temp_df = pd.DataFrame({'message': [msg]})
    temp_df = add_engineered_features(temp_df)
    temp_df['clean_text'] = temp_df['message'].apply(clean_message)
    
    t_vec = vectorizer.transform(temp_df['clean_text'])
    n_vec = csr_matrix(temp_df[num_cols].values)
    X_sample = hstack([t_vec, n_vec])
    
    pred = best_model.predict(X_sample)[0]
    label = 'SPAM' if pred == 1 else 'HAM'
    prob = best_model.predict_proba(X_sample)[0][pred] * 100
    print(f'[{label} - {prob:.1f}%] {msg}')

test_samples = [
    "Congratulations! You have won a £2,000 cash prize. Call 0800123456 now to claim!",
    "Hey, let's meet up at the library around 4pm for the NLP project discussion.",
    "URGENT: Your bank account is locked due to suspicious activity. Visit http://verify-secure.me immediately.",
    "Can you please send me the notes from yesterday's machine learning lecture?"
]

for s in test_samples:
    test_sms(s)